## Environment Setup

In [1]:
import sys
from pathlib import Path
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 1. Resolve project root directory
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Explicitly load .env from project root
env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

# 3. Imports and setup
import warnings
from core.state import ProjectState
from agents.agent_01_requirements import run_requirements_agent

warnings.filterwarnings("ignore", category=UserWarning)
print(f"Environment variables loaded from: {env_path}")




import json
from pathlib import Path
from core.state import ProjectState


def save_checkpoint(
    state: ProjectState, filepath: str = "state_checkpoint.json"
):
    """Saves current ProjectState to disk."""
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(state.model_dump_json(indent=2))
    print(f"Saved state checkpoint to {filepath}")


def load_checkpoint(filepath: str = "state_checkpoint.json") -> ProjectState:
    """Loads ProjectState from disk."""
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"No checkpoint found at {filepath}")
    with open(path, "r", encoding="utf-8") as f:
        state = ProjectState.model_validate_json(f.read())
    print(f"Loaded state checkpoint from {filepath}")
    return state

Environment variables loaded from: /workspaces/codespaces-blank/sdlc-multiagent-automation/.env


### Connection to LLM Trials

In [3]:
llm = init_chat_model(
    model="qwen/qwen3.6-27b",
    model_provider="groq",
    temperature=0,
    max_tokens=500  # Reduces requested output tokens below the 1000 limit
)

response = llm.invoke("Who created LangGraph?")
print(response.content)


<think>
Thinking Process:

1.  **Identify the core entity and question:** The user is asking about the creator of "LangGraph".
2.  **Retrieve knowledge about LangGraph:**
    *   What is LangGraph? It's a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.
    *   Who created it? LangChain (the company) created it. Specifically, it's part of the LangChain ecosystem.
    *   Who founded LangChain? Harrison Chase.
    *   Let's double-check if there's a specific individual credited with *LangGraph* specifically, or if it's just attributed to the LangChain team/company. Usually, it's attributed to the LangChain team, led by Harrison Chase.
    *   Let's do a quick mental check or search if needed (though I should rely on internal knowledge). LangGraph was introduced by the LangChain team to handle cyclic graphs and stateful multi-agent workflows, extending LangChain's capabilities.
    *   So, the creator is the LangChain team / LangChain Inc., f

In [4]:
from core.llm_factory import get_llm

llm = get_llm(schema=None)

print("Fallbacks object:")
print(llm)

for i, fb in enumerate(llm.fallbacks):
    print(i, type(fb))


=== LLM DEBUG ===
Primary: ChatGroq
Fallback 1: ChatOpenAI
Fallback 2: ChatGoogleGenerativeAI
Fallback 3: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

Fallbacks object:
runnable=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7833c2eeb4d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7833c2eebed0>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=Secre

In [2]:
from core.llm_factory import get_llm

llm = get_llm(schema=None)

response = llm.invoke("hello")

print(response.content)


=== LLM DEBUG ===
Primary: ChatGroq
Fallback 1: ChatOpenAI
Fallback 2: ChatGoogleGenerativeAI
Fallback 3: ChatGoogleGenerativeAI
Returned Type: RunnableWithFallbacks

Hello! How can I help you today?


In [4]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()
for model in models.data:
    print(model.id)

groq/compound-mini
whisper-large-v3
canopylabs/orpheus-v1-english
allam-2-7b
groq/compound
openai/gpt-oss-120b
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b


### Load Current State

In [2]:
state = load_checkpoint()

Loaded state checkpoint from state_checkpoint.json


## Agent 01 Code

In [4]:
%load_ext autoreload
%autoreload 2

# Pass whatever custom prompt you want to test
user_prompt = "Add aadhaar_no as VARCHAR(12) required field and alternate_phone as VARCHAR(20)"

state = ProjectState(raw_requirement=user_prompt)

In [ ]:
state = run_requirements_agent(state)

# Interactively view the updated Pydantic schema contract in RAM
# Display results
print('--- AGENT 1 OUTPUT ---')
print(f'Entity: {state.current_schema.entity_name}')
print(f'Total Attributes: {len(state.current_schema.attributes)}')
print('\nNewly Added Fields:')
for attr in state.current_schema.attributes[-2:]:
    print(f' - Name: {attr.name} | Type: {attr.data_type} | Required: {attr.is_required}')


save_checkpoint(state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


--- AGENT 1 OUTPUT ---
Entity: Party
Total Attributes: 13

Newly Added Fields:
 - Name: aadhaar_no | Type: VARCHAR(12) | Required: True
 - Name: alternate_phone | Type: VARCHAR(20) | Required: False
Saved state checkpoint to state_checkpoint.json


## Agent 02 Code

In [3]:
from agents.agent_02_ui import run_ui_agent

# 1. Agent 01 updates the Pydantic schema in memory
#state = run_requirements_agent(state)

# 2. Agent 02 takes the updated schema and generates Streamlit code
state = run_ui_agent(state)

print("--- AGENT 2 OUTPUT GENERATED ---")
print(f"Generated UI Code Length: {len(state.ui_code)} characters")

save_checkpoint(state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


--- AGENT 2 OUTPUT GENERATED ---
Generated UI Code Length: 16679 characters
Saved state checkpoint to state_checkpoint.json


In [13]:
# Print the exact Streamlit code Agent 2 generated
print(state.ui_code)

import streamlit as st
import re
from datetime import date

st.set_page_config(page_title="Party Master Form", layout="wide")
st.title("Banking Party Master - Party Form")

# Initialize session state for form data
if 'party_data' not in st.session_state:
    st.session_state.party_data = {}

# Define validation functions
def validate_phone_number(phone):
    if not phone:
        return True
    # Allow digits, spaces, hyphens, and plus signs
    return bool(re.match(r'^[\d\s\-\+]+$', phone))

def validate_email(email):
    if not email:
        return True
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

def validate_aadhaar(aadhaar):
    if not aadhaar:
        return False
    # Aadhaar should be exactly 12 digits
    return bool(re.match(r'^\d{12}$', aadhaar))

def validate_tax_id(tax_id):
    if not tax_id:
        return True
    # Basic validation: alphanumeric and hyphens
    return bool(re.match(r'^[a-zA-Z0-9\-]+$', t

## Agent 03 Code

In [14]:
from agents.agent_03_etl import run_etl_agent

# 1. Run requirements -> UI -> ETL sequential flow
#state = run_requirements_agent(state)
#state = run_ui_agent(state)
state = run_etl_agent(state)

print("--- AGENT 3 OUTPUT GENERATED ---")
print(f"ETL Code Length: {len(state.etl_code)} characters\n")
print(state.etl_code)

save_checkpoint(state)

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 55.3926165s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'qu

## Agent 04 Code

In [ ]:
from agents.agent_04_mdm import run_mdm_agent
from pathlib import Path

# Run Agent 04 against the active state in memory
state = run_mdm_agent(state)

# Write output to schema.sql
schema_file = Path("schema.sql")
if state.mdm_ddl:
    schema_file.write_text(state.mdm_ddl)
    print("Successfully generated schema.sql!\n")
    print("--- GENERATED SQL DDL ---")
    print(state.mdm_ddl)
else:
    print("Errors occurred:", state.errors)

save_checkpoint(state)

Errors occurred: ["Agent 04 Exception: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\\nPlease retry in 58.7900101s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM